# 20 · Stage 3 v6-A — VideoMamba-M MASK, two-stage adaptation

목표는 **v5-D acceleration 설계를 유지한 채 V-JEPA 2.1-B를 VideoMamba-M으로 교체**하는 것이다.

이번 버전은 backbone 교체 직후 feature mismatch를 줄이기 위해 **2-stage optimization**을 사용한다.

### Stage A — interface adaptation
- 1 epoch
- 4,000 micro-steps = 1,000 optimizer updates
- VideoMamba-M **전체 freeze + eval**
- 학습: `SpatialMomentPooler`, `head.project`, BiGRU/기존 heads, accel-state head

### Stage B — last-2 fine-tuning
- 5 epochs
- epoch당 4,000 micro-steps = 1,000 optimizer updates
- VideoMamba layer 30/31 + final norm unfreeze
- 마지막 2 Mamba layer는 activation checkpointing
- 나머지 backbone은 계속 freeze + eval

공통:
- batch 2 × grad accumulation 4 = effective batch 8
- input `32 × 288 × 384`
- v5-D data/sampler/loss/auxiliary design 고정
- inference lock: stride 8 / center floor .25 / accel weight 0 / state weight 0
- 최종 checkpoint 선택은 **Stage B acceleration proxy 중심**

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(
            ["git", "-C", str(REPO), "checkout", BRANCH],
            check=True,
        )
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

# Bootstrap the exact V6-A source/config when this notebook is run before
# those files are committed to stage3-sangchun. Existing repo files win.
BOOTSTRAP_SOURCE = 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nimport math\nfrom pathlib import Path\nfrom typing import Sequence\n\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\n\nfrom .v5_models import DenseTemporalCANHeadV5, SpatialMomentPooler\n\n\ndef _positive_increasing(values: Sequence[float] | None, *, name: str) -> tuple[float, ...]:\n    out = tuple(float(x) for x in (values or ()))\n    if not out:\n        raise ValueError(f"{name} must be non-empty")\n    if any(x <= 0 for x in out):\n        raise ValueError(f"{name} must contain positive values: {out}")\n    if any(b <= a for a, b in zip(out, out[1:])):\n        raise ValueError(f"{name} must be strictly increasing: {out}")\n    return out\n\n\n@dataclass(frozen=True)\nclass VideoMambaFinetuneReport:\n    depth: int\n    trainable_layer_indices: tuple[int, ...]\n    train_final_norm: bool\n    trainable_backbone_params: int\n    frozen_backbone_params: int\n\n\nclass VideoMambaDenseTokenAdapter(nn.Module):\n    """Expose dense VideoMamba patch tokens as [B,T,H,W,C]."""\n\n    def __init__(self, backbone: nn.Module) -> None:\n        super().__init__()\n        self.backbone = backbone\n        self._ft_layer_indices: tuple[int, ...] = ()\n        self._train_final_norm = False\n        self._checkpoint_trainable = False\n        self._partial_ft_configured = False\n\n    @property\n    def embed_dim(self) -> int:\n        return int(self.backbone.embed_dim)\n\n    def _spatial_pos_embed(self, h_tokens: int, w_tokens: int, *, dtype, device):\n        pos = self.backbone.pos_embed\n        extra = pos[:, :1]\n        patch = pos[:, 1:]\n        n = int(patch.shape[1])\n        side = int(round(math.sqrt(n)))\n        if side * side != n:\n            raise RuntimeError(f"VideoMamba checkpoint pos grid is not square: {n}")\n\n        if side != int(h_tokens) or side != int(w_tokens):\n            patch = patch.reshape(1, side, side, -1).permute(0, 3, 1, 2)\n            patch = F.interpolate(\n                patch.float(),\n                size=(int(h_tokens), int(w_tokens)),\n                mode="bicubic",\n                align_corners=False,\n            )\n            patch = (\n                patch.permute(0, 2, 3, 1)\n                .reshape(1, int(h_tokens) * int(w_tokens), -1)\n                .to(dtype=pos.dtype)\n            )\n        return torch.cat([extra, patch], dim=1).to(device=device, dtype=dtype)\n\n    def _temporal_pos_embed(self, t_tokens: int, *, dtype, device):\n        pos = self.backbone.temporal_pos_embedding\n        if int(pos.shape[1]) != int(t_tokens):\n            pos = F.interpolate(\n                pos.float().transpose(1, 2),\n                size=int(t_tokens),\n                mode="linear",\n                align_corners=False,\n            ).transpose(1, 2).to(dtype=self.backbone.temporal_pos_embedding.dtype)\n        return pos.to(device=device, dtype=dtype)\n\n    def configure_frozen_backbone(self) -> VideoMambaFinetuneReport:\n        """Freeze the entire VideoMamba backbone for interface adaptation."""\n        self.backbone.requires_grad_(False)\n        self._ft_layer_indices = ()\n        self._train_final_norm = False\n        self._checkpoint_trainable = False\n        self._partial_ft_configured = True\n        self.train(self.training)\n\n        frozen = sum(\n            p.numel()\n            for p in self.backbone.parameters()\n            if not p.requires_grad\n        )\n        return VideoMambaFinetuneReport(\n            depth=len(self.backbone.layers),\n            trainable_layer_indices=(),\n            train_final_norm=False,\n            trainable_backbone_params=0,\n            frozen_backbone_params=int(frozen),\n        )\n\n    def configure_partial_backbone(\n        self,\n        *,\n        last_n_layers: int = 2,\n        train_final_norm: bool = True,\n        checkpoint_trainable: bool = False,\n    ) -> VideoMambaFinetuneReport:\n        layers = self.backbone.layers\n        depth = len(layers)\n        n = int(last_n_layers)\n        if n < 1 or n > depth:\n            raise ValueError(f"last_n_layers must be in [1,{depth}], got {n}")\n\n        self.backbone.requires_grad_(False)\n        indices = tuple(range(depth - n, depth))\n        for idx in indices:\n            layers[idx].requires_grad_(True)\n        if train_final_norm:\n            self.backbone.norm_f.requires_grad_(True)\n\n        self._ft_layer_indices = indices\n        self._train_final_norm = bool(train_final_norm)\n        self._checkpoint_trainable = bool(checkpoint_trainable)\n        self._partial_ft_configured = True\n        self.train(self.training)\n\n        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)\n        frozen = sum(p.numel() for p in self.backbone.parameters() if not p.requires_grad)\n        return VideoMambaFinetuneReport(\n            depth=depth,\n            trainable_layer_indices=indices,\n            train_final_norm=bool(train_final_norm),\n            trainable_backbone_params=int(trainable),\n            frozen_backbone_params=int(frozen),\n        )\n\n    def train(self, mode: bool = True):\n        super().train(mode)\n        if self._partial_ft_configured:\n            self.backbone.eval()\n            if mode:\n                for idx in self._ft_layer_indices:\n                    self.backbone.layers[idx].train(True)\n                if self._train_final_norm:\n                    self.backbone.norm_f.train(True)\n        return self\n\n    def _run_layers(self, hidden_states, residual):\n        layers = self.backbone.layers\n        first_trainable = (\n            min(self._ft_layer_indices)\n            if self._partial_ft_configured\n            else len(layers)\n        )\n\n        if first_trainable > 0:\n            with torch.no_grad():\n                for idx in range(first_trainable):\n                    hidden_states, residual = layers[idx](\n                        hidden_states, residual, inference_params=None\n                    )\n            hidden_states = hidden_states.detach()\n            if residual is not None:\n                residual = residual.detach()\n\n        for idx in range(first_trainable, len(layers)):\n            hidden_states, residual = layers[idx](\n                hidden_states,\n                residual,\n                inference_params=None,\n                use_checkpoint=(\n                    self.training and self._checkpoint_trainable\n                ),\n            )\n        return hidden_states, residual\n\n    def forward(self, video: torch.Tensor) -> torch.Tensor:\n        if video.ndim != 5:\n            raise ValueError(f"expected [B,C,T,H,W], got {tuple(video.shape)}")\n\n        x = self.backbone.patch_embed(video)\n        b, c, t, h, w = x.shape\n        x = x.permute(0, 2, 3, 4, 1).reshape(b * t, h * w, c)\n\n        spatial_pos = self._spatial_pos_embed(h, w, dtype=x.dtype, device=x.device)\n        patch_pos = spatial_pos[:, 1:]\n        cls = (\n            self.backbone.cls_token\n            + spatial_pos[:, :1].to(dtype=self.backbone.cls_token.dtype)\n        ).expand(b, -1, -1).to(dtype=x.dtype)\n\n        x = x + patch_pos\n        x = x.reshape(b, t, h * w, c).permute(0, 2, 1, 3).reshape(b * h * w, t, c)\n        x = x + self._temporal_pos_embed(t, dtype=x.dtype, device=x.device)\n        x = x.reshape(b, h * w, t, c).permute(0, 2, 1, 3).reshape(b, t * h * w, c)\n\n        hidden_states = self.backbone.pos_drop(torch.cat([cls, x], dim=1))\n        residual = None\n        hidden_states, residual = self._run_layers(hidden_states, residual)\n\n        if residual is None:\n            residual = hidden_states\n        else:\n            residual = residual + self.backbone.drop_path(hidden_states)\n        hidden_states = self.backbone.norm_f(\n            residual.to(dtype=self.backbone.norm_f.weight.dtype)\n        )\n\n        dense = hidden_states[:, 1:]\n        expected = int(t * h * w)\n        if dense.shape[1] != expected:\n            raise RuntimeError(\n                f"unexpected VideoMamba token count {dense.shape[1]}, expected {expected}"\n            )\n        return dense.reshape(b, t, h, w, c)\n\n\nclass VideoMambaDenseCANV6A(nn.Module):\n    """VideoMamba backbone with the V5-D Stage3 decision stack."""\n\n    def __init__(\n        self,\n        backbone: nn.Module,\n        *,\n        feature_dim: int = 384,\n        temporal_hidden: int = 256,\n        temporal_layers: int = 2,\n        spatial_grid: tuple[int, int] = (3, 4),\n        spatial_gate_init: float = 0.10,\n        accel_ordinal_thresholds_mps2: Sequence[float],\n        accel_fusion_enabled: bool,\n        accel_fusion_hidden: int,\n        accel_fusion_gate_init: float,\n        accel_fusion_detach_ordinal_inputs: bool,\n        accel_state_thresholds_mps2: Sequence[float],\n        accel_state_hidden: int = 96,\n        accel_state_dropout: float = 0.10,\n        stop_thresholds_mps: Sequence[float],\n        turn_yaw_thresholds_rps: Sequence[float],\n        steer_activity_thresholds: Sequence[float],\n        brake_thresholds_bar: Sequence[float],\n        throttle_thresholds_pct: Sequence[float],\n    ) -> None:\n        super().__init__()\n        self.backbone = VideoMambaDenseTokenAdapter(backbone)\n        embed_dim = int(self.backbone.embed_dim)\n\n        self.spatial_pool = SpatialMomentPooler(\n            embed_dim,\n            grid_size=spatial_grid,\n            gate_init=spatial_gate_init,\n        )\n        self.head = DenseTemporalCANHeadV5(\n            input_dim=embed_dim,\n            feature_dim=int(feature_dim),\n            hidden=int(temporal_hidden),\n            layers=int(temporal_layers),\n            accel_ordinal_thresholds_mps2=accel_ordinal_thresholds_mps2,\n            accel_fusion_enabled=bool(accel_fusion_enabled),\n            accel_fusion_hidden=int(accel_fusion_hidden),\n            accel_fusion_gate_init=float(accel_fusion_gate_init),\n            accel_fusion_detach_ordinal_inputs=bool(accel_fusion_detach_ordinal_inputs),\n            stop_thresholds_mps=stop_thresholds_mps,\n            turn_yaw_thresholds_rps=turn_yaw_thresholds_rps,\n            steer_activity_thresholds=steer_activity_thresholds,\n            brake_thresholds_bar=brake_thresholds_bar,\n            throttle_thresholds_pct=throttle_thresholds_pct,\n        )\n\n        self.accel_state_thresholds_mps2 = _positive_increasing(\n            accel_state_thresholds_mps2,\n            name="accel_state_thresholds_mps2",\n        )\n\n        ordinal_thresholds = tuple(float(x) for x in self.head.accel_ordinal_thresholds_mps2)\n        stop_thresholds = tuple(float(x) for x in self.head.stop_thresholds_mps)\n        state_feature_dim = 5 + 2 * len(ordinal_thresholds) + len(stop_thresholds)\n\n        hidden = int(accel_state_hidden)\n        self.accel_state_head = nn.Sequential(\n            nn.LayerNorm(state_feature_dim),\n            nn.Linear(state_feature_dim, hidden),\n            nn.GELU(),\n            nn.Dropout(float(accel_state_dropout)),\n            nn.Linear(hidden, len(self.accel_state_thresholds_mps2) * 3),\n        )\n        final = self.accel_state_head[-1]\n        assert isinstance(final, nn.Linear)\n        nn.init.normal_(final.weight, mean=0.0, std=0.01)\n        nn.init.zeros_(final.bias)\n\n    def configure_frozen_backbone(self):\n        return self.backbone.configure_frozen_backbone()\n\n    def configure_partial_backbone(\n        self,\n        *,\n        last_n_layers=2,\n        train_final_norm=True,\n        checkpoint_trainable=False,\n    ):\n        return self.backbone.configure_partial_backbone(\n            last_n_layers=last_n_layers,\n            train_final_norm=train_final_norm,\n            checkpoint_trainable=checkpoint_trainable,\n        )\n\n    def forward(self, video: torch.Tensor) -> dict[str, torch.Tensor]:\n        z = self.spatial_pool(self.backbone(video))\n        outputs = self.head(z)\n\n        speed = outputs["speed_mps"]\n        speed_delta = torch.cat(\n            [torch.zeros_like(speed[:, :1]), speed[:, 1:] - speed[:, :-1]],\n            dim=1,\n        )\n        fused_accel = outputs["accel_from_speed_mps2"]\n        raw_accel = outputs.get("accel_raw_from_speed_mps2", fused_accel)\n        explicit_delta = outputs["delta_speed_mps"]\n        ordinal_prob = torch.sigmoid(outputs["accel_ordinal_logits"]).flatten(start_dim=2)\n        stop_prob = torch.sigmoid(outputs["stop_ordinal_logits"])\n\n        state_features = torch.cat(\n            [\n                speed.unsqueeze(-1),\n                speed_delta.unsqueeze(-1),\n                fused_accel.unsqueeze(-1),\n                raw_accel.unsqueeze(-1),\n                (explicit_delta / 0.10).unsqueeze(-1),\n                ordinal_prob,\n                stop_prob,\n            ],\n            dim=-1,\n        )\n        logits = self.accel_state_head(state_features).view(\n            state_features.shape[0],\n            state_features.shape[1],\n            len(self.accel_state_thresholds_mps2),\n            3,\n        )\n        outputs["accel_state_logits"] = logits\n        outputs["accel_state_thresholds_mps2"] = logits.new_tensor(\n            self.accel_state_thresholds_mps2, dtype=torch.float32\n        )\n        return outputs\n\n\ndef _checkpoint_state(checkpoint_path: str | Path):\n    obj = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=False)\n    if isinstance(obj, dict) and isinstance(obj.get("model"), dict):\n        return dict(obj["model"]), obj\n    if isinstance(obj, dict):\n        return dict(obj), {}\n    raise TypeError(f"unsupported checkpoint payload: {type(obj)!r}")\n\n\ndef load_v5d_compatible_warm_start(\n    model: VideoMambaDenseCANV6A,\n    checkpoint_path: str | Path,\n) -> dict:\n    """Copy V5-D tensors whose semantics and shapes are unchanged."""\n    state, meta = _checkpoint_state(checkpoint_path)\n    current = model.state_dict()\n    fresh_prefixes = ("backbone.", "spatial_pool.", "head.project.")\n\n    copied = {}\n    skipped_fresh = []\n    skipped_shape = {}\n    missing = []\n\n    for key, value in state.items():\n        if key.startswith(fresh_prefixes):\n            skipped_fresh.append(key)\n            continue\n        if key not in current:\n            missing.append(key)\n            continue\n        if tuple(value.shape) != tuple(current[key].shape):\n            skipped_shape[key] = {\n                "source": tuple(value.shape),\n                "target": tuple(current[key].shape),\n            }\n            continue\n        copied[key] = value\n\n    incompatible = model.load_state_dict(copied, strict=False)\n    return {\n        "checkpoint_epoch": meta.get("epoch"),\n        "checkpoint_best_score": meta.get("best_score"),\n        "copied_tensors": len(copied),\n        "copied_keys": sorted(copied),\n        "skipped_fresh": sorted(skipped_fresh),\n        "skipped_shape": skipped_shape,\n        "missing_source_keys": sorted(missing),\n        "missing_after_partial_load": sorted(incompatible.missing_keys),\n        "unexpected_after_partial_load": sorted(incompatible.unexpected_keys),\n    }\n\n\ndef split_v6a_optimizer_parameters(\n    model: VideoMambaDenseCANV6A,\n    *,\n    require_backbone: bool = True,\n):\n    groups = {\n        "interface": [],\n        "head": [],\n        "accel_state_head": [],\n        "backbone_penultimate": [],\n        "backbone_last": [],\n        "backbone_final_norm": [],\n    }\n\n    layers = model.backbone.backbone.layers\n    depth = len(layers)\n    penultimate_prefix = f"backbone.backbone.layers.{depth - 2}."\n    last_prefix = f"backbone.backbone.layers.{depth - 1}."\n    norm_prefix = "backbone.backbone.norm_f."\n\n    for name, p in model.named_parameters():\n        if not p.requires_grad:\n            continue\n        if name.startswith("spatial_pool.") or name.startswith("head.project."):\n            groups["interface"].append((name, p))\n        elif name.startswith("accel_state_head."):\n            groups["accel_state_head"].append((name, p))\n        elif name.startswith(penultimate_prefix):\n            groups["backbone_penultimate"].append((name, p))\n        elif name.startswith(last_prefix):\n            groups["backbone_last"].append((name, p))\n        elif name.startswith(norm_prefix):\n            groups["backbone_final_norm"].append((name, p))\n        elif name.startswith("head."):\n            groups["head"].append((name, p))\n        else:\n            raise RuntimeError(f"unclassified trainable V6-A parameter: {name}")\n\n    always_required = ("interface", "head", "accel_state_head")\n    for key in always_required:\n        if not groups[key]:\n            raise RuntimeError(f"V6-A optimizer family is empty: {key}")\n\n    backbone_keys = (\n        "backbone_penultimate",\n        "backbone_last",\n        "backbone_final_norm",\n    )\n    if require_backbone:\n        for key in backbone_keys:\n            if not groups[key]:\n                raise RuntimeError(\n                    f"V6-A optimizer family is empty: {key}"\n                )\n    else:\n        for key in backbone_keys:\n            if groups[key]:\n                raise RuntimeError(\n                    "Stage-A frozen backbone unexpectedly has trainable "\n                    f"parameters in {key}"\n                )\n    return groups\n\n\n__all__ = [\n    "VideoMambaFinetuneReport",\n    "VideoMambaDenseTokenAdapter",\n    "VideoMambaDenseCANV6A",\n    "load_v5d_compatible_warm_start",\n    "split_v6a_optimizer_parameters",\n]\n'
BOOTSTRAP_CONFIG = 'seed: 20260918\nexperiment:\n  name: videomamba_m_mask_can_v6a_backbone_ablation\n  baseline_run: vjepa21b_can_v5d_accel_decision\n  baseline_checkpoint: best_accel_proxy.pt\n  purpose: backbone_only_ablation\ndata:\n  clip_len: 32\n  train_window_stride: 16\n  val_window_stride: 16\n  input_height: 288\n  input_width: 384\n  num_workers: 2\n  val_num_workers: 0\n  train_random_flip: true\n  sampler:\n    inverse_frequency_power: 0.35\n    max_normalized_weight: 30.0\n    event_multipliers:\n      cruise: 0.55\n      turn: 1.5\n      moderate_accel: 2.75\n      moderate_decel: 3.25\n      moderate_mixed: 3.75\n      hard_accel: 4.25\n      hard_decel: 4.75\n      hard_mixed: 5.0\n      stop_start: 4.0\n      reversal: 2.0\n    source_multipliers:\n      comma2k19: 1.0\n      a2d2: 20.0\n  sources:\n    comma2k19:\n      steering_direction_deadzone_deg: 2.0\n      steering_activity_scale_deg: 8.0\n      steering_sign_multiplier: 1.0\n    a2d2:\n      steering_direction_deadzone_deg: 10.0\n      steering_activity_scale_deg: 60.0\n      steering_sign_multiplier: 1.0\nmodel:\n  backbone: videomamba_middle\n  official_repo: OpenGVLab/VideoMamba\n  official_commit: 37355c26d0ae99ca2459f6d4044a5f509031a79f\n  official_checkpoint_repo: OpenGVLab/VideoMamba\n  official_checkpoint: videomamba_m16_k400_mask_ft_f32_res224.pth\n  official_pretraining: masked_video_pretraining_then_Kinetics-400\n  official_input_frames: 32\n  official_resolution: 224\n  embed_dim: 576\n  depth: 32\n  tubelet_size: 1\n  temporal_hidden: 256\n  temporal_layers: 2\n  feature_dim: 384\n  spatial_pool:\n    grid:\n    - 3\n    - 4\n    gate_init: 0.1\n  accel_ordinal_thresholds_mps2:\n  - 0.1\n  - 0.2\n  - 0.3\n  - 0.5\n  accel_fusion:\n    enabled: true\n    hidden: 64\n    gate_init: 0.1\n    detach_ordinal_inputs: true\n  accel_state:\n    thresholds_mps2:\n    - 0.1\n    - 0.2\n    - 0.3\n    - 0.5\n    hidden: 96\n    dropout: 0.1\n  stop_thresholds_mps:\n  - 0.1\n  - 0.3\n  - 0.5\n  - 1.0\n  - 2.0\n  turn_yaw_thresholds_rps:\n  - 0.01\n  - 0.03\n  - 0.05\n  steer_activity_thresholds:\n  - 0.25\n  - 0.5\n  - 1.0\n  - 2.0\n  brake_thresholds_bar:\n  - 0.5\n  - 2.0\n  - 5.0\n  throttle_thresholds_pct:\n  - 1.0\n  - 5.0\n  - 10.0\n  backbone_finetune:\n    last_n_layers: 2\n    train_final_norm: true\n    checkpoint_trainable: true\ntraining:\n  epochs: 6\n  batch_size: 2\n  grad_accum_steps: 4\n  interface_learning_rate: 5.0e-05\n  head_learning_rate: 1.0e-05\n  accel_state_head_learning_rate: 5.0e-05\n  penultimate_layer_learning_rate: 1.0e-06\n  last_layer_learning_rate: 2.0e-06\n  final_norm_learning_rate: 2.0e-06\n  interface_weight_decay: 0.01\n  head_weight_decay: 0.01\n  accel_state_head_weight_decay: 0.01\n  backbone_weight_decay: 0.05\n  warmup_ratio: 0.1\n  min_learning_rate_ratio: 0.1\n  grad_clip_norm: 1.0\n  amp_dtype: bf16\n  max_steps_per_epoch: 4000\n  max_val_steps: 800\n  early_stopping_patience: 0\n  log_interval: 20\n  two_stage:\n    enabled: true\n    stage_a:\n      name: interface_adaptation\n      epochs: 1\n      max_steps_per_epoch: 4000\n      early_stopping_patience: 0\n      freeze_backbone: true\n    stage_b:\n      name: last2_finetune\n      epochs: 5\n      max_steps_per_epoch: 4000\n      early_stopping_patience: 0\n      last_n_layers: 2\n      train_final_norm: true\n      checkpoint_trainable: true\nwarm_start:\n  run_name: vjepa21b_can_v5d_accel_decision\n  checkpoint: best_accel_proxy.pt\n  mode: compatible_downstream\nloss:\n  base:\n    mode: accel_v4_fusion\n    weights:\n      speed_mps: 1.0\n      accel_from_speed_mps2: 1.5\n      steering_deg: 1.5\n      yaw_rate_rps: 0.75\n    normalization: {}\n    accel_v2:\n      magnitude_weighted_regression:\n        scale_mps2: 0.3\n        gain: 2.0\n        max_weight: 3.0\n        beta: 0.5\n      multi_threshold_margin:\n        weight: 0.9\n        thresholds_mps2:\n        - 0.1\n        - 0.2\n        - 0.3\n        - 0.5\n        temperature_mps2: 0.05\n      speed_delta:\n        weight: 0.6\n        beta_normalized: 0.01\n      speed_accel_consistency:\n        weight: 0.15\n        speed_delta_scale_normalized: 0.01\n        accel_scale_normalized: 0.5\n        beta: 0.5\n    accel_v3:\n      ordinal:\n        weight: 0.75\n        thresholds_mps2:\n        - 0.1\n        - 0.2\n        - 0.3\n        - 0.5\n        monotonic_weight: 0.1\n    accel_v4:\n      fusion:\n        diagnostics_only: true\n  stop_ordinal:\n    weight: 0.5\n    thresholds_mps:\n    - 0.1\n    - 0.3\n    - 0.5\n    - 1.0\n    - 2.0\n    monotonic_weight: 0.1\n  delta_speed:\n    weight: 0.75\n    beta_mps: 0.05\n    dt_s: 0.1\n    accel_consistency_weight: 0.2\n    consistency_beta_mps2: 0.2\n  accel_state:\n    weight: 0.75\n    thresholds_mps2:\n    - 0.1\n    - 0.2\n    - 0.3\n    - 0.5\n    label_smoothing: 0.02\n    monotonic_weight: 0.05\n  accel_temporal_gradient:\n    weight: 0.15\n    horizons:\n    - 1\n    - 2\n    - 4\n    beta_normalized: 0.1\n  integrated_kinematics:\n    weight: 0.3\n    horizons:\n    - 1\n    - 2\n    - 4\n    dt_s: 0.1\n    beta_mps: 0.05\n  turn_ordinal:\n    weight: 0.25\n    thresholds_rps:\n    - 0.01\n    - 0.03\n    - 0.05\n    monotonic_weight: 0.1\n  steer_direction:\n    weight: 0.2\n  steer_activity_ordinal:\n    weight: 0.1\n    thresholds:\n    - 0.25\n    - 0.5\n    - 1.0\n    - 2.0\n    monotonic_weight: 0.1\n  brake_ordinal:\n    weight: 0.1\n    thresholds_bar:\n    - 0.5\n    - 2.0\n    - 5.0\n    monotonic_weight: 0.1\n  throttle_ordinal:\n    weight: 0.05\n    thresholds_pct:\n    - 1.0\n    - 5.0\n    - 10.0\n    monotonic_weight: 0.1\nvalidation:\n  proxy_rules:\n    sensitive:\n      stop_speed_mps: 0.3\n      accel_deadzone_mps2: 0.1\n      steer_deadzone_deg: 2.0\n    medium:\n      stop_speed_mps: 0.5\n      accel_deadzone_mps2: 0.2\n      steer_deadzone_deg: 5.0\n    conservative:\n      stop_speed_mps: 1.0\n      accel_deadzone_mps2: 0.3\n      steer_deadzone_deg: 8.0\n  acceptance:\n    min_accel_delta_vs_v5d_anchor: 0.005\n    min_medium_accelerating_delta: 0.0\n    max_dynamic_to_constant_delta: 0.0\ninference_lock:\n  clip_len: 32\n  stride: 8\n  center_floor: 0.25\n  stop_weight: 0.75\n  accel_weight: 0.0\n  steer_weight: 0.3\n  turn_weight: 0.3\n  state_weight: 0.0\nlogging:\n  wandb_enabled: true\n  wandb_project: blackbox-stage3\n  wandb_group: videomamba_m_mask_can_v6a\n'

src_target = REPO / "src/blackbox_detection/stage3/v6a_videomamba.py"
cfg_target = REPO / "configs/stage3/videomamba_m_mask_can_v6a.yaml"
src_target.parent.mkdir(parents=True, exist_ok=True)
cfg_target.parent.mkdir(parents=True, exist_ok=True)

src_target.write_text(BOOTSTRAP_SOURCE, encoding="utf-8")
cfg_target.write_text(BOOTSTRAP_CONFIG, encoding="utf-8")
print("synced generated source:", src_target)
print("synced generated config:", cfg_target)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "wandb==0.29.0",
        "easydict==1.13",
        "huggingface-hub==0.34.4",
        "ninja",
        "packaging",
        "wheel",
    ],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils import (
    dataloader_seed_kwargs,
    finish_wandb,
    init_wandb,
    seed_everything,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
A2D2_ROOT = DATA_ROOT / "A2D2" / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests/stage3/v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs/stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
WANDB_KEY_PATH = DRIVE_ROOT / "wandb_key.txt"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_OUTPUT_ROOT = Path("/content/stage3_runs")
for p in [OUTPUT_ROOT, PRETRAINED_ROOT, LOCAL_PRETRAINED_ROOT, LOCAL_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = cfg_target
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

loss_cfg = copy.deepcopy(cfg["loss"])
loss_cfg["base"]["normalization"] = {
    name: {
        "mean": float(stats[name]["mean"]),
        "std": float(stats[name]["std"]),
    }
    for name in ("speed_mps", "accel_from_speed_mps2")
}

print("Python         :", sys.version)
print("Torch          :", torch.__version__)
print("CUDA           :", torch.version.cuda)
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Git            :", BRANCH, GIT_COMMIT)
print("Config         :", CFG_PATH)
print("metric contract: PASS")


## 1. Pin official VideoMamba and ensure the authors' BiMamba extension

공식 VideoMamba는 일반 최신 `mamba-ssm`과 달리 `Mamba(..., bimamba=True)`를 사용한다.
따라서 arbitrary PyPI 최신판으로 대체하지 않고, **공식 repo에 포함된 causal-conv1d /
mamba fork**를 우선 사용한다.

이미 현재 runtime에서 `bimamba=True` smoke가 통과하면 compile은 건너뛴다.
그렇지 않으면 pinned official repo의 두 extension을 editable install한다.

In [ ]:
VIDEO_MAMBA_REPO = Path("/content/VideoMamba")
vm_cfg = cfg["model"]
VIDEO_MAMBA_COMMIT = str(vm_cfg["official_commit"])

if not (VIDEO_MAMBA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/OpenGVLab/VideoMamba.git",
         str(VIDEO_MAMBA_REPO)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(VIDEO_MAMBA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VIDEO_MAMBA_REPO), "checkout", "-q", VIDEO_MAMBA_COMMIT],
    check=True,
)

ACTUAL_VM_COMMIT = subprocess.run(
    ["git", "-C", str(VIDEO_MAMBA_REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert ACTUAL_VM_COMMIT == VIDEO_MAMBA_COMMIT

def bimamba_subprocess_smoke() -> bool:
    probe = subprocess.run(
        [
            sys.executable, "-c",
            (
                "import torch; "
                "from mamba_ssm.modules.mamba_simple import Mamba; "
                "m=Mamba(d_model=16, expand=2, bimamba=True); "
                "print(type(m).__name__)"
            ),
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print("BiMamba probe failed:")
        print(probe.stderr[-4000:])
        return False
    print("BiMamba probe:", probe.stdout.strip())
    return True

if not bimamba_subprocess_smoke():
    env = dict(os.environ)
    env["MAX_JOBS"] = "4"
    for package_dir in (
        VIDEO_MAMBA_REPO / "causal-conv1d",
        VIDEO_MAMBA_REPO / "mamba",
    ):
        print("installing official extension:", package_dir)
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install",
                "--no-build-isolation", "-e", str(package_dir),
            ],
            check=True,
            env=env,
        )

    if not bimamba_subprocess_smoke():
        raise RuntimeError(
            "Official VideoMamba BiMamba extension still fails after "
            "pinned local install. Do not substitute standard mamba-ssm: "
            "it is not checkpoint-compatible with bimamba=True."
        )

print("VideoMamba commit:", ACTUAL_VM_COMMIT)


## 2. Download official VideoMamba-M MASK K400 32-frame checkpoint

선정 이유:
- **32 frames**: 현재 Stage3 `T=32`와 정확히 일치
- **Middle / 576-d / 32 layers**: V-JEPA-B와 capacity 차이를 Small보다 줄임
- **masked video pretraining + K400 fine-tuning**: motion-sensitive representation을 성능 우선으로 사용
- 최종 DACON 제출 ZIP 제한은 10GB이므로 model-size 자체는 병목이 아님

공식 checkpoint는 224×224에서 fine-tune되었지만, V6-A dense adapter가 positional
embedding을 실제 Stage3의 18×24 patch grid(288×384)에 bicubic interpolation한다.


In [ ]:
from huggingface_hub import hf_hub_download

def _is_usable_file(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= int(min_bytes)
    except OSError:
        return False

def copy_file_to_local(
    source: Path,
    destination: Path,
    *,
    min_bytes: int = 1,
):
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_name(destination.name + ".copy.tmp")
    tmp.unlink(missing_ok=True)
    with source.open("rb") as src, tmp.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    if tmp.stat().st_size < int(min_bytes):
        raise OSError(f"staged file too small: {tmp.stat().st_size}")
    os.replace(tmp, destination)

VM_CKPT_NAME = str(vm_cfg["official_checkpoint"])
VM_CKPT_DRIVE = PRETRAINED_ROOT / VM_CKPT_NAME
VM_CKPT_LOCAL = LOCAL_PRETRAINED_ROOT / VM_CKPT_NAME

if not _is_usable_file(VM_CKPT_DRIVE, 10_000_000):
    downloaded = Path(
        hf_hub_download(
            repo_id=str(vm_cfg["official_checkpoint_repo"]),
            filename=VM_CKPT_NAME,
        )
    )
    copy_file_to_local(
        downloaded,
        VM_CKPT_DRIVE,
        min_bytes=10_000_000,
    )

if not _is_usable_file(VM_CKPT_LOCAL, 10_000_000):
    copy_file_to_local(
        VM_CKPT_DRIVE,
        VM_CKPT_LOCAL,
        min_bytes=10_000_000,
    )

print("VideoMamba checkpoint:", VM_CKPT_LOCAL)
print("size MiB:", VM_CKPT_LOCAL.stat().st_size / 2**20)


## 3. Build the exact V5-D data/sampler protocol

event cache도 v5-D와 동일한 `v5d/event_index_t32_s16.npz`를 재사용한다.

In [ ]:
from torch.utils.data import DataLoader, Subset

from blackbox_detection.stage3.v5_dataset import build_event_balanced_sampler
from blackbox_detection.stage3.v5d_dataset import MixedStage3CANV5DDataset

dc = cfg["data"]
tc = cfg["training"]
source_cfg = dc["sources"]

a2d2_manifest = pd.read_csv(A2D2_ROOT / "manifest.csv")
assert int(a2d2_manifest["num_frames"].sum()) == 9151

train_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_train.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
    {
        "name": "a2d2",
        "manifest": a2d2_manifest,
        "processed_root": A2D2_ROOT,
        **source_cfg["a2d2"],
    },
]
val_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_val_id.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
]

train_ds = MixedStage3CANV5DDataset(
    train_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["train_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=dc["train_random_flip"],
    seed=SEED,
)
val_ds = MixedStage3CANV5DDataset(
    val_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["val_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=False,
    seed=SEED + 1,
)

sampler_cfg = dc["sampler"]
EVENT_CACHE = (
    DRIVE_ROOT / "manifests/stage3/v5d"
    / f"event_index_t{dc['clip_len']}_s{dc['train_window_stride']}.npz"
)

train_sampler, sampler_report = build_event_balanced_sampler(
    train_ds,
    num_samples=int(tc["max_steps_per_epoch"]) * int(tc["batch_size"]),
    event_multipliers=sampler_cfg["event_multipliers"],
    source_multipliers=sampler_cfg["source_multipliers"],
    inverse_frequency_power=sampler_cfg["inverse_frequency_power"],
    max_normalized_weight=sampler_cfg["max_normalized_weight"],
    seed=SEED,
    cache_path=EVENT_CACHE,
)

n_val_eval = min(int(tc["max_val_steps"]), len(val_ds))
val_indices = np.linspace(
    0, len(val_ds) - 1,
    num=n_val_eval,
    dtype=np.int64,
)
val_eval_ds = Subset(
    val_ds,
    np.unique(val_indices).tolist(),
)

train_loader = DataLoader(
    train_ds,
    batch_size=tc["batch_size"],
    sampler=train_sampler,
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED),
)
val_workers = int(dc.get("val_num_workers", 0))
val_loader = DataLoader(
    val_eval_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=val_workers,
    pin_memory=True,
    persistent_workers=val_workers > 0,
    **dataloader_seed_kwargs(SEED + 1),
)

assert len(val_eval_ds) == int(tc["max_val_steps"])
print("event cache          :", EVENT_CACHE)
print("train windows        :", len(train_ds))
print("sampled train / epoch:", len(train_sampler))
print("validation windows   :", len(val_eval_ds))
print("A2D2 expected share  :", sampler_report["expected_source_share"]["a2d2"])


## 4. Construct VideoMamba-M and load official MASK→K400 weights

공식 fine-tuning script와 맞춰 `videomamba_middle`, 32 frames, tubelet=1,
drop-path=0.4로 architecture를 만든다. Classification head는 downstream에서
사용하지 않지만 checkpoint integrity 확인을 위해 400-class head까지 load한다.


In [ ]:
import importlib.util

VIDEO_SM = VIDEO_MAMBA_REPO / "videomamba/video_sm"
if str(VIDEO_SM) not in sys.path:
    sys.path.insert(0, str(VIDEO_SM))

vm_file = VIDEO_SM / "models/videomamba.py"
spec = importlib.util.spec_from_file_location(
    "official_videomamba_model",
    vm_file,
)
vm_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(vm_module)

official_backbone = vm_module.videomamba_middle(
    pretrained=False,
    img_size=int(vm_cfg["official_resolution"]),
    num_classes=400,
    drop_path_rate=0.4,
    kernel_size=int(vm_cfg["tubelet_size"]),
    num_frames=int(vm_cfg["official_input_frames"]),
    use_checkpoint=False,
    checkpoint_num=0,
)

raw = torch.load(
    VM_CKPT_LOCAL,
    map_location="cpu",
    weights_only=False,
)
if isinstance(raw, dict):
    state = None
    for key in ("model", "module", "state_dict"):
        if isinstance(raw.get(key), dict):
            state = dict(raw[key])
            break
    if state is None:
        state = dict(raw)
else:
    raise TypeError(type(raw))

state = {
    (k[7:] if k.startswith("module.") else k): v
    for k, v in state.items()
}

msg = official_backbone.load_state_dict(state, strict=False)
allowed_missing = set()
allowed_unexpected = set()
if set(msg.missing_keys) - allowed_missing:
    raise RuntimeError(f"official checkpoint missing keys: {msg.missing_keys}")
if set(msg.unexpected_keys) - allowed_unexpected:
    raise RuntimeError(f"official checkpoint unexpected keys: {msg.unexpected_keys}")

print("official checkpoint load: PASS")
print("embed dim:", official_backbone.embed_dim)
print("depth    :", len(official_backbone.layers))
assert int(official_backbone.embed_dim) == int(vm_cfg["embed_dim"]) == 576
assert len(official_backbone.layers) == int(vm_cfg["depth"]) == 32


## 5. Build V6-A + V5-D compatible warm start

VideoMamba-M과 V-JEPA-B의 channel dimension이 다르므로 아래만 fresh로 둔다.

- VideoMamba-M backbone
- 576-d `SpatialMomentPooler`
- backbone-dependent `head.project`

아래는 V5-D E2에서 warm-start한다.

- BiGRU temporal stack
- continuous CAN heads
- acceleration ordinal/fusion
- STOP / turn / steering / brake / throttle auxiliary heads
- V5-D 3-state acceleration head

Stage A와 Stage B는 **서로 다른 checkpoint/optimizer directory**를 사용한다.
Stage A의 optimizer state가 Stage B에 섞이지 않는다.

In [ ]:
from blackbox_detection.stage3.v6a_videomamba import (
    VideoMambaDenseCANV6A,
    load_v5d_compatible_warm_start,
    split_v6a_optimizer_parameters,
)
from blackbox_detection.stage3.trainer import build_scheduler
from blackbox_detection.stage3.v5d_trainer import V5DTrainer

mc = cfg["model"]
tc = cfg["training"]
two_stage = tc["two_stage"]
stage_a_cfg = dict(two_stage["stage_a"])
stage_b_cfg = dict(two_stage["stage_b"])

fusion_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])
state_cfg = dict(mc["accel_state"])

model = VideoMambaDenseCANV6A(
    official_backbone,
    feature_dim=mc["feature_dim"],
    temporal_hidden=mc["temporal_hidden"],
    temporal_layers=mc["temporal_layers"],
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=spatial_cfg["gate_init"],
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=fusion_cfg["enabled"],
    accel_fusion_hidden=fusion_cfg["hidden"],
    accel_fusion_gate_init=fusion_cfg["gate_init"],
    accel_fusion_detach_ordinal_inputs=fusion_cfg["detach_ordinal_inputs"],
    accel_state_thresholds_mps2=state_cfg["thresholds_mps2"],
    accel_state_hidden=state_cfg["hidden"],
    accel_state_dropout=state_cfg["dropout"],
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

RUN_VARIANT = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT
LOCAL_RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_VARIANT

STAGE_A_DIR = RUN_DIR / "stage_a_interface"
LOCAL_STAGE_A_DIR = LOCAL_RUN_DIR / "stage_a_interface"

for p in (RUN_DIR, LOCAL_RUN_DIR, STAGE_A_DIR, LOCAL_STAGE_A_DIR):
    p.mkdir(parents=True, exist_ok=True)

def restore_persistent_checkpoints(persistent_dir, local_dir):
    for filename in (
        "latest.pt",
        "best.pt",
        "best_proxy.pt",
        "best_accel_proxy.pt",
        "history.csv",
    ):
        persistent = persistent_dir / filename
        local = local_dir / filename
        if (
            not local.is_file()
            and persistent.is_file()
        ):
            min_bytes = 1_000_000 if filename.endswith(".pt") else 1
            copy_file_to_local(
                persistent,
                local,
                min_bytes=min_bytes,
            )

restore_persistent_checkpoints(STAGE_A_DIR, LOCAL_STAGE_A_DIR)
restore_persistent_checkpoints(RUN_DIR, LOCAL_RUN_DIR)

stage_a_resume = LOCAL_STAGE_A_DIR / "latest.pt"
stage_b_resume = LOCAL_RUN_DIR / "latest.pt"

warm_report = None
if not stage_a_resume.is_file() and not stage_b_resume.is_file():
    wc = cfg["warm_start"]
    source_drive = OUTPUT_ROOT / wc["run_name"] / wc["checkpoint"]
    source_local = (
        LOCAL_PRETRAINED_ROOT
        / f"{wc['run_name']}__{wc['checkpoint']}"
    )
    if not _is_usable_file(source_local, 1_000_000):
        if not _is_usable_file(source_drive, 1_000_000):
            raise FileNotFoundError(source_drive)
        copy_file_to_local(
            source_drive,
            source_local,
            min_bytes=1_000_000,
        )

    warm_report = load_v5d_compatible_warm_start(
        model,
        source_local,
    )
    if warm_report["unexpected_after_partial_load"]:
        raise RuntimeError(
            warm_report["unexpected_after_partial_load"]
        )

    print("V5-D compatible warm start: PASS")
    print("source epoch  :", warm_report["checkpoint_epoch"])
    print("copied tensors:", warm_report["copied_tensors"])
else:
    print("Existing V6-A checkpoint found; V5-D one-time warm start skipped.")

frozen_report = model.configure_frozen_backbone()
assert frozen_report.depth == 32
assert frozen_report.trainable_layer_indices == ()
assert frozen_report.trainable_backbone_params == 0

print("Stage A backbone config:", frozen_report)


## 6. Phase-specific optimizer builders

두 단계는 optimizer와 scheduler를 완전히 분리한다.

- Stage A: interface + downstream heads만 optimizer에 포함
- Stage B: 위 항목 + Mamba layer 30/31 + final norm 포함

두 단계 모두 effective batch는 8이며 각 epoch에 1,000 optimizer updates다.

In [ ]:
def add_decay_groups(
    out,
    *,
    family_name,
    named_params,
    lr,
    weight_decay,
):
    decay, no_decay = [], []
    for name, p in named_params:
        if p.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    if decay:
        out.append({
            "params": decay,
            "lr": float(lr),
            "weight_decay": float(weight_decay),
            "name": f"{family_name}_decay",
        })
    if no_decay:
        out.append({
            "params": no_decay,
            "lr": float(lr),
            "weight_decay": 0.0,
            "name": f"{family_name}_no_decay",
        })


def build_phase_optimizer_and_scheduler(
    model,
    *,
    phase: str,
    phase_cfg: dict,
):
    is_stage_b = phase == "stage_b"
    families = split_v6a_optimizer_parameters(
        model,
        require_backbone=is_stage_b,
    )

    groups = []
    add_decay_groups(
        groups,
        family_name="interface",
        named_params=families["interface"],
        lr=tc["interface_learning_rate"],
        weight_decay=tc["interface_weight_decay"],
    )
    add_decay_groups(
        groups,
        family_name="head",
        named_params=families["head"],
        lr=tc["head_learning_rate"],
        weight_decay=tc["head_weight_decay"],
    )
    add_decay_groups(
        groups,
        family_name="accel_state",
        named_params=families["accel_state_head"],
        lr=tc["accel_state_head_learning_rate"],
        weight_decay=tc["accel_state_head_weight_decay"],
    )

    if is_stage_b:
        add_decay_groups(
            groups,
            family_name="mamba30",
            named_params=families["backbone_penultimate"],
            lr=tc["penultimate_layer_learning_rate"],
            weight_decay=tc["backbone_weight_decay"],
        )
        add_decay_groups(
            groups,
            family_name="mamba31",
            named_params=families["backbone_last"],
            lr=tc["last_layer_learning_rate"],
            weight_decay=tc["backbone_weight_decay"],
        )
        add_decay_groups(
            groups,
            family_name="mamba_norm",
            named_params=families["backbone_final_norm"],
            lr=tc["final_norm_learning_rate"],
            weight_decay=tc["backbone_weight_decay"],
        )

    optimizer = torch.optim.AdamW(groups)

    micro_steps_per_epoch = min(
        len(train_loader),
        int(phase_cfg["max_steps_per_epoch"]),
    )
    optimizer_steps_per_epoch = max(
        math.ceil(
            micro_steps_per_epoch
            / int(tc["grad_accum_steps"])
        ),
        1,
    )
    total_steps = (
        optimizer_steps_per_epoch
        * int(phase_cfg["epochs"])
    )

    scheduler = build_scheduler(
        optimizer,
        total_steps=total_steps,
        warmup_ratio=tc["warmup_ratio"],
        min_ratio=tc["min_learning_rate_ratio"],
    )
    return optimizer, scheduler, optimizer_steps_per_epoch


def print_optimizer(optimizer, *, phase):
    print(f"\n[{phase}]")
    for group in optimizer.param_groups:
        print(
            group["name"],
            "lr=", group["lr"],
            "wd=", group["weight_decay"],
            "params=", sum(p.numel() for p in group["params"]),
        )


## 7. W&B

Stage A와 Stage B를 하나의 W&B run에 기록하되 checkpoint는 분리한다.

In [ ]:
lc = cfg.get("logging", {})
WANDB_ENABLED = bool(lc.get("wandb_enabled", True))
wandb_run = None

if WANDB_ENABLED:
    import wandb

    if not WANDB_KEY_PATH.is_file():
        raise FileNotFoundError(WANDB_KEY_PATH)
    wandb.login(
        key=WANDB_KEY_PATH.read_text(encoding="utf-8").strip(),
        relogin=False,
    )
    finish_wandb()

    run_id_path = RUN_DIR / "wandb_run_id.txt"
    stored_run_id = (
        run_id_path.read_text(encoding="utf-8").strip()
        if run_id_path.is_file()
        else None
    )
    stored_run_id = stored_run_id or None

    wandb_run = init_wandb(
        project=str(lc.get("wandb_project", "blackbox-stage3")),
        entity=os.getenv("WANDB_ENTITY") or None,
        name=f"{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}",
        group=str(lc.get("wandb_group", "videomamba_m_mask_can_v6a")),
        tags=[
            "stage3",
            "v6a",
            "videomamba-middle",
            "masked-pretrain",
            "two-stage",
            "interface-adaptation",
            "last2-finetune",
            "acceleration",
            "a2d2",
            "t32",
            "event-balanced",
            "kinematics",
        ],
        run_id=stored_run_id,
        resume="allow",
        config={
            "git_commit": GIT_COMMIT,
            "videomamba_commit": ACTUAL_VM_COMMIT,
            "run_variant": RUN_VARIANT,
            "seed": SEED,
            "two_stage": two_stage,
            "frozen_report": frozen_report.__dict__,
            "sampler_report": sampler_report,
            "warm_start_report": warm_report,
        },
    )
    if not stored_run_id and wandb_run is not None:
        run_id_path.write_text(
            str(wandb_run.id),
            encoding="utf-8",
        )

vc = cfg["validation"]
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
AMP_DTYPE = (
    torch.bfloat16
    if str(tc["amp_dtype"]).lower() in {"bf16", "bfloat16"}
    else torch.float16
)
model.to(DEVICE)

print("device:", DEVICE)
print("two-stage schedule:", json.dumps(two_stage, indent=2))


## 8. Full-resolution two-stage smoke

실제 입력 `1×3×32×288×384`으로 두 단계를 각각 backward한다.

### Stage A 확인
- VideoMamba backbone 전체 gradient 없음
- `SpatialMomentPooler`, `head.project`, accel-state head에는 gradient 존재

### Stage B 확인
- layer 29 gradient 없음
- layer 30/31 gradient 존재
- fresh interface/accel-state head도 계속 gradient 존재

이 셀에서 OOM이면 본 학습을 시작하지 않는다.

In [ ]:
from blackbox_detection.stage3.v5d_accel import v5d_multitask_loss

def grad_norm(module):
    grads = [
        p.grad.detach().float().norm()
        for p in module.parameters()
        if p.grad is not None
    ]
    return (
        float(torch.stack(grads).norm().cpu())
        if grads else 0.0
    )


smoke = next(iter(train_loader))
video = smoke["video"][:1].to(DEVICE)
target = smoke["target"][:1].to(DEVICE)
valid = smoke["valid"][:1].to(DEVICE)
aux = {
    k: v[:1].to(DEVICE)
    for k, v in smoke["aux"].items()
}

# -------------------------
# Stage A smoke
# -------------------------
model.configure_frozen_backbone()
opt_a, _, _ = build_phase_optimizer_and_scheduler(
    model,
    phase="stage_a",
    phase_cfg=stage_a_cfg,
)
opt_a.zero_grad(set_to_none=True)
model.train()

assert model.backbone.backbone.training is False
assert not any(
    p.requires_grad
    for p in model.backbone.backbone.parameters()
)

with torch.autocast(
    device_type=DEVICE.type,
    dtype=AMP_DTYPE,
    enabled=DEVICE.type == "cuda",
):
    out_a = model(video)
    loss_a, _ = v5d_multitask_loss(
        out_a,
        target,
        valid,
        aux,
        loss_cfg,
        stats=stats,
    )

assert torch.isfinite(loss_a)
loss_a.backward()

assert all(
    p.grad is None
    for p in model.backbone.backbone.parameters()
)
for name, value in {
    "spatial": grad_norm(model.spatial_pool),
    "project": grad_norm(model.head.project),
    "state": grad_norm(model.accel_state_head),
}.items():
    assert value > 0 and np.isfinite(value), (name, value)

print("Stage A full-res backward: PASS")
opt_a.zero_grad(set_to_none=True)
del opt_a, out_a, loss_a

# -------------------------
# Stage B smoke
# -------------------------
ft_report = model.configure_partial_backbone(
    last_n_layers=int(stage_b_cfg["last_n_layers"]),
    train_final_norm=bool(stage_b_cfg["train_final_norm"]),
    checkpoint_trainable=bool(
        stage_b_cfg["checkpoint_trainable"]
    ),
)
assert ft_report.depth == 32
assert ft_report.trainable_layer_indices == (30, 31)

opt_b, _, _ = build_phase_optimizer_and_scheduler(
    model,
    phase="stage_b",
    phase_cfg=stage_b_cfg,
)
opt_b.zero_grad(set_to_none=True)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model.train()
layers = model.backbone.backbone.layers
assert model.backbone.backbone.training is False
assert layers[29].training is False
assert layers[30].training is True
assert layers[31].training is True

with torch.autocast(
    device_type=DEVICE.type,
    dtype=AMP_DTYPE,
    enabled=DEVICE.type == "cuda",
):
    out_b = model(video)
    loss_b, _ = v5d_multitask_loss(
        out_b,
        target,
        valid,
        aux,
        loss_cfg,
        stats=stats,
    )

assert torch.isfinite(loss_b)
assert tuple(out_b["accel_state_logits"].shape) == (
    1,
    dc["clip_len"],
    4,
    3,
)
loss_b.backward()

assert all(
    p.grad is None
    for p in layers[29].parameters()
)

for name, value in {
    "layer30": grad_norm(layers[30]),
    "layer31": grad_norm(layers[31]),
    "spatial": grad_norm(model.spatial_pool),
    "project": grad_norm(model.head.project),
    "state": grad_norm(model.accel_state_head),
}.items():
    assert value > 0 and np.isfinite(value), (name, value)

peak_gib = (
    torch.cuda.max_memory_allocated() / 2**30
    if DEVICE.type == "cuda"
    else 0.0
)

print("Stage B full-res backward: PASS")
print("Stage B peak GPU GiB:", peak_gib)

opt_b.zero_grad(set_to_none=True)
del opt_b, video, target, valid, aux, out_b, loss_b
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 9. Load fixed v5-D baseline diagnostics

V6-A의 목표는 Stage3 전체가 아니라 **acceleration이 backbone 교체로 개선되는지**다.
따라서 동일 800-window protocol의 v5-D E2 summary를 baseline으로 고정한다.

In [ ]:
BASELINE_RUN = OUTPUT_ROOT / cfg["experiment"]["baseline_run"]
BASELINE_SUMMARY = BASELINE_RUN / "summary.json"
if not BASELINE_SUMMARY.is_file():
    raise FileNotFoundError(BASELINE_SUMMARY)

baseline = json.loads(
    BASELINE_SUMMARY.read_text(encoding="utf-8")
)

baseline_metrics = {
    "stage3": float(baseline["diagnostic_best_proxy_stage3"]),
    "accel": float(baseline["diagnostic_best_proxy_accel"]),
    "steer": float(baseline["diagnostic_best_proxy_steer"]),
    "medium_accelerating": float(
        baseline["best_medium_f1_accelerating"]
    ),
    "medium_dynamic_to_constant": float(
        baseline["best_medium_dynamic_to_constant_rate"]
    ),
}
print(json.dumps(baseline_metrics, indent=2))


## 10. Train / resume — Stage A → Stage B

Stage A와 B의 checkpoint directory가 분리되어 있으므로 resume이 안전하다.

- Stage A: `.../stage_a_interface/latest.pt`
- Stage B: `.../latest.pt`

Stage A가 이미 끝났다면 checkpoint를 load하고 즉시 Stage B로 넘어간다.
Stage B가 중단된 경우에는 Stage B `latest.pt`에서 이어간다.

In [ ]:
def make_trainer(
    *,
    optimizer,
    scheduler,
    local_dir,
    sync_dir,
    phase_name,
):
    phase_config = copy.deepcopy(cfg)
    phase_config["training_phase"] = phase_name
    return V5DTrainer(
        model,
        optimizer,
        scheduler=scheduler,
        device=DEVICE,
        grad_accum_steps=tc["grad_accum_steps"],
        grad_clip_norm=tc["grad_clip_norm"],
        amp_dtype=tc["amp_dtype"],
        loss_weights=loss_cfg,
        stats=stats,
        proxy_rules=vc["proxy_rules"],
        output_dir=local_dir,
        sync_dir=sync_dir,
        wandb_enabled=WANDB_ENABLED,
        log_interval=tc["log_interval"],
        config=phase_config,
    )


# ============================================================
# Stage A: frozen VideoMamba, interface adaptation
# ============================================================
stage_a_resume = LOCAL_STAGE_A_DIR / "latest.pt"
stage_b_resume = LOCAL_RUN_DIR / "latest.pt"

if stage_b_resume.is_file():
    print(
        "Stage B checkpoint already exists; "
        "Stage A execution is skipped."
    )
    history_a = []
    stage_a_history_csv = LOCAL_STAGE_A_DIR / "history.csv"
    if stage_a_history_csv.is_file():
        stage_a_flat = pd.read_csv(stage_a_history_csv)
        print(
            "Recovered Stage A history rows:",
            len(stage_a_flat),
        )
else:
    model.configure_frozen_backbone()

    optimizer_a, scheduler_a, updates_a = (
        build_phase_optimizer_and_scheduler(
            model,
            phase="stage_a",
            phase_cfg=stage_a_cfg,
        )
    )
    print_optimizer(optimizer_a, phase="Stage A")
    print(
        "Stage A optimizer updates/epoch:",
        updates_a,
    )

    trainer_a = make_trainer(
        optimizer=optimizer_a,
        scheduler=scheduler_a,
        local_dir=LOCAL_STAGE_A_DIR,
        sync_dir=STAGE_A_DIR,
        phase_name="stage_a_interface_adaptation",
    )

    history_a = trainer_a.fit(
        train_loader,
        val_loader,
        epochs=int(stage_a_cfg["epochs"]),
        max_train_steps=int(
            stage_a_cfg["max_steps_per_epoch"]
        ),
        max_val_steps=None,
        resume_from=(
            stage_a_resume
            if stage_a_resume.is_file()
            else None
        ),
        early_stopping_patience=int(
            stage_a_cfg["early_stopping_patience"]
        ),
        backfill_validation_on_resume=False,
    )

    if not (LOCAL_STAGE_A_DIR / "latest.pt").is_file():
        raise RuntimeError(
            "Stage A did not produce latest.pt"
        )

    print("Stage A COMPLETE")

# ============================================================
# Stage B: unfreeze last 2 Mamba layers + final norm
# ============================================================
ft_report = model.configure_partial_backbone(
    last_n_layers=int(stage_b_cfg["last_n_layers"]),
    train_final_norm=bool(stage_b_cfg["train_final_norm"]),
    checkpoint_trainable=bool(
        stage_b_cfg["checkpoint_trainable"]
    ),
)
assert ft_report.trainable_layer_indices == (30, 31)

optimizer_b, scheduler_b, updates_b = (
    build_phase_optimizer_and_scheduler(
        model,
        phase="stage_b",
        phase_cfg=stage_b_cfg,
    )
)
print_optimizer(optimizer_b, phase="Stage B")
print("Stage B optimizer updates/epoch:", updates_b)

trainer_b = make_trainer(
    optimizer=optimizer_b,
    scheduler=scheduler_b,
    local_dir=LOCAL_RUN_DIR,
    sync_dir=RUN_DIR,
    phase_name="stage_b_last2_finetune",
)

stage_b_resume = LOCAL_RUN_DIR / "latest.pt"
if (
    not stage_b_resume.is_file()
    and history_a
):
    trainer_b.global_step = int(
        history_a[-1].get("global_step") or 0
    )

history_b = trainer_b.fit(
    train_loader,
    val_loader,
    epochs=int(stage_b_cfg["epochs"]),
    max_train_steps=int(
        stage_b_cfg["max_steps_per_epoch"]
    ),
    max_val_steps=None,
    resume_from=(
        stage_b_resume
        if stage_b_resume.is_file()
        else None
    ),
    early_stopping_patience=int(
        stage_b_cfg["early_stopping_patience"]
    ),
    backfill_validation_on_resume=False,
)

def flatten_phase_history(history, phase):
    rows = []
    for row in history:
        rows.append({
            "phase": phase,
            "epoch": row["epoch"],
            "minutes": row["minutes"],
            "learning_rate": row.get("learning_rate"),
            "global_step": row.get("global_step"),
            "max_gpu_memory_gib": row.get(
                "max_gpu_memory_gib"
            ),
            **{
                f"train/{k}": v
                for k, v in row["train"].items()
            },
            **{
                f"val/{k}": v
                for k, v in row["val"].items()
            },
        })
    return rows

combined_rows = []
if history_a:
    combined_rows += flatten_phase_history(
        history_a,
        "stage_a",
    )
else:
    stage_a_csv = LOCAL_STAGE_A_DIR / "history.csv"
    if stage_a_csv.is_file():
        a_df = pd.read_csv(stage_a_csv)
        a_df.insert(0, "phase", "stage_a")
        combined_rows += a_df.to_dict("records")

combined_rows += flatten_phase_history(
    history_b,
    "stage_b",
)
combined_history_df = pd.DataFrame(combined_rows)
display(combined_history_df)

combined_path = (
    LOCAL_RUN_DIR / "history_two_stage.csv"
)
combined_history_df.to_csv(
    combined_path,
    index=False,
)
trainer_b._sync_file(combined_path)

print("Stage B COMPLETE")


## 11. Acceleration-first Stage B summary / gate

최종 판단은 **Stage B만** 사용한다. Stage A는 feature-space adaptation 과정이므로
최종 checkpoint 후보와 섞지 않는다.

Stage B의 `best_accel_proxy.pt`가 overlap validation의 기본 후보이다.

In [ ]:
stage_b_df = combined_history_df[
    combined_history_df["phase"] == "stage_b"
].reset_index(drop=True)

if not len(stage_b_df):
    raise RuntimeError("empty Stage B history")

accel_col = "val/proxy/robust_mean_accel_macro_f1"
stage3_col = "val/proxy/robust_mean_stage3_score"
steer_col = "val/proxy/robust_mean_steer_macro_f1"
acc_cls_col = "val/proxy/medium/f1_accel_ACCELERATING"
d2c_col = "val/proxy/medium/dynamic_to_constant_rate"

idx = pd.to_numeric(
    stage_b_df[accel_col],
    errors="coerce",
).idxmax()
row = stage_b_df.loc[idx]

current = {
    "stage_b_epoch": int(row["epoch"]),
    "stage3": float(row[stage3_col]),
    "accel": float(row[accel_col]),
    "steer": float(row[steer_col]),
    "medium_accelerating": float(
        row[acc_cls_col]
    ),
    "medium_dynamic_to_constant": float(
        row[d2c_col]
    ),
}

delta = {
    key: current[key] - baseline_metrics[key]
    for key in (
        "stage3",
        "accel",
        "steer",
        "medium_accelerating",
        "medium_dynamic_to_constant",
    )
}

ac = vc["acceptance"]
checks = {
    "accel_plus_0p005": (
        delta["accel"]
        >= float(
            ac["min_accel_delta_vs_v5d_anchor"]
        )
    ),
    "accelerating_nonnegative": (
        delta["medium_accelerating"]
        >= float(
            ac["min_medium_accelerating_delta"]
        )
    ),
    "dynamic_to_constant_nonpositive": (
        delta["medium_dynamic_to_constant"]
        <= float(
            ac["max_dynamic_to_constant_delta"]
        )
    ),
}
local_accept = all(checks.values())

stage_a_last = history_a[-1] if history_a else None
stage_a_last_val = (
    stage_a_last.get("val")
    if stage_a_last is not None
    else None
)
if stage_a_last_val is None:
    stage_a_rows = combined_history_df[
        combined_history_df["phase"] == "stage_a"
    ]
    if len(stage_a_rows):
        last_a = stage_a_rows.iloc[-1]
        stage_a_last_val = {
            key[4:]: float(value)
            for key, value in last_a.items()
            if (
                isinstance(key, str)
                and key.startswith("val/")
                and pd.notna(value)
            )
        }

summary = {
    "run_variant": RUN_VARIANT,
    "git_commit": GIT_COMMIT,
    "videomamba_commit": ACTUAL_VM_COMMIT,
    "official_checkpoint": VM_CKPT_NAME,
    "backbone": "VideoMamba-M MASK",
    "optimization": {
        "type": "two_stage",
        "effective_batch": (
            int(tc["batch_size"])
            * int(tc["grad_accum_steps"])
        ),
        "stage_a": {
            "epochs": int(stage_a_cfg["epochs"]),
            "max_steps_per_epoch": int(
                stage_a_cfg["max_steps_per_epoch"]
            ),
            "backbone": "fully_frozen",
        },
        "stage_b": {
            "epochs": int(stage_b_cfg["epochs"]),
            "max_steps_per_epoch": int(
                stage_b_cfg["max_steps_per_epoch"]
            ),
            "trainable_layers": list(
                ft_report.trainable_layer_indices
            ),
            "train_final_norm": bool(
                stage_b_cfg["train_final_norm"]
            ),
        },
    },
    "clip_len": int(dc["clip_len"]),
    "input_size": [
        int(dc["input_height"]),
        int(dc["input_width"]),
    ],
    "warm_start": {
        "run": cfg["warm_start"]["run_name"],
        "checkpoint": cfg["warm_start"]["checkpoint"],
        "copied_tensors": (
            warm_report["copied_tensors"]
            if warm_report is not None
            else None
        ),
    },
    "v5d_baseline": baseline_metrics,
    "stage_a_last_val": stage_a_last_val,
    "best_accel_stage_b_epoch": (
        current["stage_b_epoch"]
    ),
    "v6a_best_accel": current,
    "delta_vs_v5d": delta,
    "acceptance_checks": checks,
    "local_accept_for_overlap_validation": local_accept,
    "inference_lock": cfg["inference_lock"],
    "note": (
        "Final selection uses Stage B only. "
        "Local proxy is not DACON leaderboard. "
        "If accepted, run fixed V5-C overlap protocol "
        "without tuning state_weight."
    ),
}

local_summary = LOCAL_RUN_DIR / "summary.json"
local_summary.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)
trainer_b._sync_file(local_summary)

print(json.dumps(summary, indent=2))

if WANDB_ENABLED:
    finish_wandb()

print("Stage A latest             :", STAGE_A_DIR / "latest.pt")
print("Stage B latest             :", RUN_DIR / "latest.pt")
print("Stage B val-loss best      :", RUN_DIR / "best.pt")
print("Stage B proxy best         :", RUN_DIR / "best_proxy.pt")
print("Stage B accel-proxy best   :", RUN_DIR / "best_accel_proxy.pt")
print("next overlap validation?   :", local_accept)


## 다음 단계

`summary.json`과 `history_two_stage.csv`를 확인한다.

- `local_accept_for_overlap_validation = false`  
  → Stage B까지 충분히 적응시켰는데도 VideoMamba-M MASK가 acceleration 병목을 해결하지 못한 것.

- `true`  
  → **Stage B `best_accel_proxy.pt`를 고정**하고 v5-C와 동일한 25 complete segments /
  T32 / stride8 / center_floor=.25 / production fusion / `state_weight=0`으로
  `21_validate_v6a_overlap.ipynb`를 실행한다.

공개 50 labels는 이 학습 notebook에서 사용하지 않는다.

최종 DACON 제출 ZIP 제한은 10GB라 VideoMamba-M model size는 병목이 아니다.
다만 custom BiMamba/causal-conv CUDA extension은 submission export/runtime 호환성을
별도로 검증해야 한다.